# VGGish + SVM

**Модель:** SVM (RBF) на VGGish-эмбеддингах (mean + std)

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds, get_durations
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_cv_fold

import warnings
warnings.filterwarnings("ignore")

In [2]:
vggish_model = torch.hub.load("harritaylor/torchvggish", "vggish")
vggish_model.eval()
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
vggish_model = vggish_model.to(device)

def extract_vggish(path: str) -> np.ndarray:
    with torch.no_grad():
        emb = vggish_model.forward(path).cpu().numpy()
    if emb.ndim == 1:
        emb = emb[np.newaxis, :]
    return np.concatenate([emb.mean(axis=0), emb.std(axis=0)]).astype(np.float32)

Using cache found in /Users/dk/.cache/torch/hub/harritaylor_torchvggish_master


In [3]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()
print(f"Train+Val: {len(paths_trainval)}, Test: {len(paths_test)}")

Train+Val: 2356, Test: 416


In [4]:
print("Извлечение эмбеддингов (train+val)...")
X_embeds_trainval = build_feature_matrix(paths_trainval, extract_vggish, n_jobs=1)
print("Извлечение эмбеддингов (test)...")
X_embeds_test = build_feature_matrix(paths_test, extract_vggish, n_jobs=1)

del vggish_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Извлечение эмбеддингов (train+val)...
Извлечение эмбеддингов (test)...


In [5]:
path_to_idx = {p: i for i, p in enumerate(paths_trainval)}

dur_trainval = get_durations(paths_trainval).reshape(-1, 1)
dur_test = get_durations(paths_test).reshape(-1, 1)
X_trainval = np.hstack([X_embeds_trainval, letters_trainval, dur_trainval])
X_test = np.hstack([X_embeds_test, letters_test, dur_test])

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C": [0.1, 0.5, 1.0, 5.0, 10.0],
    "clf__gamma": ["scale", "auto", 0.01, 0.1],
}

with start_run("exp_vggish_svm", group="01_baselines"):

    mlflow.log_params({
        "model": "SVM RBF",
        "features": "VGGish mean+std (256-dim)",
        "cv_folds": config.CV_N_SPLITS,
        "class_weight": "balanced",
    })

    fold_results = []
    t0 = time.perf_counter()

    for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
            get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

        idx_tr  = [path_to_idx[p] for p in paths_tr]
        idx_val = [path_to_idx[p] for p in paths_val]
        X_tr  = np.hstack([X_embeds_trainval[idx_tr],  letters_tr,  get_durations(paths_tr).reshape(-1, 1)])
        X_val = np.hstack([X_embeds_trainval[idx_val], letters_val, get_durations(paths_val).reshape(-1, 1)])

        grid = GridSearchCV(pipeline, param_grid, cv=3,
                            scoring="f1_macro", refit=True, n_jobs=-1)
        grid.fit(X_tr, labels_tr)
        clf = grid.best_estimator_

        y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
        thr = find_optimal_threshold(labels_val, y_proba_val)
        metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=False)
        fold_results.append(metrics)

        print(f"Фолд {fold+1}/{config.CV_N_SPLITS}, best_params={grid.best_params_}")
        log_cv_fold(fold, f1_bad=metrics["f1_bad"], f1_macro=metrics["f1_macro"],
                    roc_auc=metrics["roc_auc"], threshold=thr)

    train_time_sec = time.perf_counter() - t0
    cv_agg = evaluate_cv_folds(fold_results)
    print_cv_summary(cv_agg)

    print("\nПодбор гиперпараметров на train+val и оценка на test:")

    final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                              scoring="f1_macro", refit=True, n_jobs=-1)
    final_grid.fit(X_trainval, labels_trainval)
    mlflow.log_params({f"best_{k}": v for k, v in final_grid.best_params_.items()})

    cv_threshold = cv_agg["threshold_mean"]
    y_proba_test = final_grid.best_estimator_.predict_proba(X_test)[:, config.CLASS_BAD]
    test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)

    save_result_csv(
        exp_dir=exp_dir,
        experiment_id="exp_vggish_svm",
        experiment_name="VGGish + SVM (baseline)",
        model="VGGish mean+std + SVM RBF",
        accuracy=test_metrics["accuracy"],
        f1_macro=test_metrics["f1_macro"],
        f1_bad=test_metrics["f1_bad"],
        roc_auc=test_metrics["roc_auc"],
        precision_bad=test_metrics["precision_bad"],
        recall_bad=test_metrics["recall_bad"],
        threshold=test_metrics["threshold"],
        cv_f1_bad_std=cv_agg["f1_bad_std"],
        cv_f1_macro_std=cv_agg["f1_macro_std"],
        cv_accuracy_std=cv_agg["accuracy_std"],
        cv_roc_auc_std=cv_agg["roc_auc_std"],
        cv_precision_bad_std=cv_agg["precision_bad_std"],
        cv_recall_bad_std=cv_agg["recall_bad_std"],
        cv_threshold_std=cv_agg["threshold_std"],
        embed_dim=256,
        embed_dim_note="VGGish 128-dim * 2 (mean+std)",
        notes=f"5-fold CV + holdout | best={final_grid.best_params_} | thr={cv_threshold:.2f}",
        train_time_sec=train_time_sec,
    )

Фолд 1/5, best_params={'clf__C': 0.5, 'clf__gamma': 'scale'}
Фолд 2/5, best_params={'clf__C': 1.0, 'clf__gamma': 'scale'}
Фолд 3/5, best_params={'clf__C': 1.0, 'clf__gamma': 'auto'}
Фолд 4/5, best_params={'clf__C': 1.0, 'clf__gamma': 'scale'}
Фолд 5/5, best_params={'clf__C': 1.0, 'clf__gamma': 'scale'}
accuracy          0.7423±0.0204
f1_macro          0.7179±0.0146
f1_bad            0.6361±0.0148
roc_auc           0.7872±0.0240
precision_bad     0.5912±0.0428
recall_bad        0.6970±0.0583
Средний threshold: 0.34

Подбор гиперпараметров на train+val и оценка на test:
              precision    recall  f1-score   support

        good       0.82      0.80      0.81       281
         bad       0.60      0.62      0.61       135

    accuracy                           0.74       416
   macro avg       0.71      0.71      0.71       416
weighted avg       0.75      0.74      0.74       416

Threshold : 0.34
Accuracy  : 0.7428
F1-macro  : 0.7094
F1-bad    : 0.6109
ROC-AUC   : 0.7962
